# HOW TO GLIMT

In [1]:
%load_ext autoreload
%autoreload 2

## Retrieve article text from article name on demand

In [2]:
import wikipediaapi

language = 'en'
user_agent = wikipediaapi.USER_AGENT
wiki = wikipediaapi.Wikipedia(user_agent,language)

title = "Lars"
page = wiki.page(title)

page.summary

'Lars is a common male name in Scandinavian countries.'

## Add Wikipedia article search lookup

### Download every Wikipedia article name

### Map all article names to latent space

In [3]:
with open('glimt/wikipedia/enwiki-20250701-all-titles-in-ns0', 'r') as f:
    titles = [x[0:-1].replace('_', ' ') for x in f.readlines()[1:]]

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch
from tqdm import tqdm

# Load model (choose one optimized for short phrases)
model = SentenceTransformer("all-MiniLM-L6-v2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from tqdm import tqdm
import numpy as np

batch_size = 1000
embeddings = []

for i in tqdm(range(0, len(titles), batch_size), total=(len(titles) + batch_size - 1) // batch_size):
    batch = titles[i:i + batch_size]
    emb = model.encode(
        batch,
        convert_to_numpy=True,
        batch_size=batch_size,
        device=device,
        show_progress_bar=False  # tqdm already shows progress
    )
    embeddings.append(emb)

# Combine into single matrix
all_embeddings = np.vstack(embeddings)


# Save
all_embeddings = np.vstack(embeddings)
np.save("wiki_title_embeddings.npy", all_embeddings)
with open("wiki_titles.txt", "w", encoding="utf-8") as f:
    f.writelines(t + "\n" for t in titles)

  0%|                                        | 9/9259 [00:06<1:48:29,  1.42it/s]

Get top 20 articles that seem relevant by title
Get the text of each article
Get top 5 articles that are minimum distance between article content and title + fine print content
Summarize long articles if necessary if oversize

### Summarize Wiki pages down to keep size uniform
### Put the wiki texts into the prompt
### Put the Wiki links into Sources
### Make rationale and wwwcwu easier to strip out of response

## Load IFPs

In [ ]:
from list_active_ifps import list_active_ifps
ifps = list_active_ifps()

In [ ]:
from save_ifps_to_disk import save_ifps_to_disk
id_to_ifp = save_ifps_to_disk(ifps)

In [ ]:
from ifps_to_df import ifps_to_df
ifps_to_df(ifps).sort_values(by='id').head(3)

## Gather news for IFPs

In [ ]:
from gather_news_for_ifps import gather_news_for_ifps
news = gather_news_for_ifps(ifps)

## Split news into text and URLs

In [ ]:
ifp = ifps[0]
ifp_id = ifp['id']
ifp_id

In [ ]:
ifp_news = news[ifp_id]

In [ ]:
s1 = ifp_news.split('\n\n')

In [ ]:
s2 = [x for x in s1 if '\nSource' in x]

In [ ]:
s3 = [x.split('\nSource:') for x in s2]

In [ ]:
ifp_news_text = [x[0].split('\nOriginal language:')[0] for x in s3]

In [ ]:
ifp_news_sources = [x.split('](')[1][0:-1] for x in [x[1] for x in s3]]

In [ ]:
ifp_news = dict([(x,y) for x,y in zip(ifp_news_sources, ifp_news_text)])

In [ ]:
ifp_news_sources = [x for x,y in ifp_news.items()]
ifp_news_text = [y for x,y in ifp_news.items()]

## Detect binary

In [ ]:
from is_binary import is_binary

In [ ]:
ifp = ifps[0]
is_binary(ifp)

## Rephrase binary outcomes

In [ ]:
from rephrase_binary_outcomes import rephrase_binary_outcomes

In [ ]:
events = rephrase_binary_outcomes(ifp)

In [ ]:
ifp['bins'][0]['props']['title'] = events[0]
ifp['bins'][1]['props']['title'] = events[1]

In [ ]:
ifp['bins']

## Rationale fields split into for and against

In [ ]:
os.makedirs('glimt/prompt', exist_ok=True)

## Summarize rationale and wwcwu separately

## Run the two first IFPs

## Look at their community alignment

## Add special cautions in text or task-specific cautions if community alignment remains poor

## Do all 19 questions

## Prompts for each question

In [ ]:
all_ifps = ifps.copy()

In [ ]:
prompt = {}

for ifp in ifps:
    id, startDay, endDay, title, details = [ifp['id'], ifp['dates']['startDay'],
              ifp['dates']['endDay'], ifp['props']['title'], ifp['props']['details']]
    fn = f'glimt/prompt/{id}.txt'
    bins = [x['props']['title'] for x in ifp['bins']]
    p1 = f"""
You are a talented, experienced and confident superforecaster. You are asked a question:

```question
{title}
```

You are given details on how to interpret the terms of the question:

```details
{details}
```

The following contemporary news is available for this question:
```news
{news[id]}
```

You must output a rationale R.  
This is a Markdown format text giving
* My rationale
* Reasons I might be right
* Reasons I might be wrong
Output this wrapped with tag
```rationale
```
"""
    if len(bins) == 2 and bins[0] == 'Yes':
        bp = f"""

This question resolves as Yes if the event happens and No otherwise.  You must output a numerical forecast which is the probability P, 0 <= P <= 1 of a yes answer wrapped in a tag in this format:
```probability_of_yes
P
```"""
    else:
        bp = f"""

This question can have one of {len(bins)} outcomes namely {', '.join(bins)}.  You must output a numerical forecast which a Python list of
{len(bins)} probabilities, where 0 <= Pi <= 1 and P1 + P2 + ... + P{len(bins)} = 1, in format
```binProbs
[P1,P2,...,P{len(bins)}]
```
"""
    bpw = "\nYou MUST ALWAYS OUTPUT A NUMERICAL FORECAST.  The task is failed if you omit this."
    prompt[id] = p1 + bp + bpw
    with open(fn, 'w') as f:
        f.write(prompt[id])

## Get 5 rolls of the LLM for each prompt

In [ ]:
from tqdm import tqdm

In [ ]:
rolls = {}
tries = 5
for ifp in tqdm(ifps):
    id = ifp['id']
    p = prompt[id]
    rolls[id] = [humor_me(p) for _ in range(tries)]

## Break out 5 forecasts and rationales for each IFP

In [ ]:
import numpy as np

In [ ]:
def get_bin_probs(r):
    if 'binProbs' in r:
        return eval(r.split('```binProbs')[1].split('```')[0].strip())
    else:
        p_yes = float(r.split('```probability_of_yes')[1].split('```')[0].strip())
        p_no = 1 - p_yes
        return [p_yes, p_no]

In [ ]:
forecasts = {}
rationales = {}

In [ ]:
for id in rolls:
    R = rolls[id]
    forecast = [get_bin_probs(r) for r in R]
    forecasts[id] = forecast

In [ ]:
def get_rationale(r):
    r1 = r.split('```rationale')[1].replace('Executive Summary of Rationale', 'Rationale').replace('Reasons You', 'Reasons I')
    r2 = r1.split('```')[0].replace('\nR\n', '\n')
    return r2.strip()

In [ ]:
for id in rolls:
    R = rolls[id]
    rats = []
    for r in R:
        rat = get_rationale(r)
        rats.append(rat)
    rationales[id] = rats

In [ ]:
forecasts

## Median forecasts and rationales

In [ ]:
def median_forecast(Fs):
    M = np.array(Fs)
    return np.median(M, axis=0).tolist()

In [ ]:
def median_rationale(Rs):

    WRs = [f"""```forecast
{x}
```""" for x in Rs]
    
    WRS = '\n'.join(WRs)
    
    prompt = f"""
Summarize the gist of the rationale or thinking of the following answers from different forecasters to a single problem. 

{WRS}

DO NOT REFER TO THE FORECASTERS OR MULTIPLE FORECASTS.
PRESENT THIS AS YOUR OWN THINKING, YOUR OWN RATIONALE.
DO NOT USE ANY KIND OF MARKDOWN SYNTAX IN THE RATIONALE.
SEPARATE into "Reasons I'm Right" and "Reasons I Could Be Wrong" sections.
Each section must be 1200 characters or less.
"""
    
    medrat = humor_me(prompt)
    
    return medrat

## Combine and reject binProbs that are too long

In [ ]:
median_forecasts = {}

for ifp in ifps:
    id = ifp['id']
    n_bins = len(ifp['bins'])
    Fs = forecasts[id]
    Rs = rationales[id]
    cum = []
    for fs, rs in zip(Fs,Rs):
        if len(fs) != n_bins:
            print('problem', id, n_bins, fs)
        else:
            cum.append((fs, rs))
    Fs = [fs for fs, rs in cum]
    Rs = [rs for fs, rs in cum]
    fors = [for_part(rs) for rs in Rs]
    againsts = [against_part(rs) for rs in Rs]
    probas = median_forecast(Fs)
    rationale = median_rationale(fors)
    wwcym = median_rational(againsts)
    urls = extract_urls(news[id]) + extract_urls(wiki[id])
    median_forecasts[id] = probas, rationale, wwcym, urls

## Format forecast upload with all the goodies

In [ ]:
def jsx_forecast(id: int, 
                 probas: list[float],
                 reason: str,
                 wwcym: str,
                 urls: list[str]):
    reason = reason[0:1200] # hard size on GUI
    wwcym = wwcym[0:1200] # hard size on GUI
    U = repr([{"value": url, "index":i+1,"order":i+1} for i, url in enumerate(urls)]).replace("'", '"')
    return f"""[["ifps","submitFcst",{{"ifpId":{id},"data":{{"probas": {probas} }},"rationale":{{"type":"reason","reason":"{reason}","wwcym":"{wwcym}","urls":{U}}},"publish":true}}]]"""

In [ ]:
id = 463
reason = "The articles indicate that both Russia and Ukraine are engaged in diplomatic efforts, with Russia offering to halt military operations and the US proposing settlement ideas."
wwcym = "The articles also indicate that Russia is unwilling to abandon its territorial demands, which could prolong the conflict and delay any ceasefire."
urls = ['presstv.ir','www.nytimes.com']
jsx_forecast(id,probas,reason,wwcym,urls)

## Submit median forecast to IFP

In [ ]:
import json

def submit_forecast(id,probas,reason,wwcym,urls):
    return jsx_request(jsx_forecast(id,probas,reason,wwcym,urls))

In [ ]:
def submit_ifp(id):
    probas, reason, wwcym, urls = median_forecasts[id]
    return submit_forecast(id,probas,reason,wwcym,urls)